# Demo: substituční šifra a kryptoanalýza

Notebook ukazuje povinné funkční API, pohodlné objektové API a finální konfiguraci použitou pro učitelské ciphertexty. Neslouží k opětovnému zpracování všech 60 ciphertextů; načítá existující výsledky z `outputs/` a provádí jen krátkou demonstrační ukázku.

Používaná abeceda je `ABCDEFGHIJKLMNOPQRSTUVWXYZ_`, kde `_` představuje mezeru.

Finální konfigurace pro dávkové dešifrování:

```text
referenční matice: data/processed/TM_ref_krakatit.npy
restarty: 2
iterace na restart: 10000
celkem iterací na ciphertext: 20000
polish_key: zapnuto
```


In [1]:
from pathlib import Path
import csv
import sys

import matplotlib.pyplot as plt
import numpy as np

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / 'src').exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

SRC_DIR = PROJECT_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from substitution_cipher import (
    ALPHABET,
    SubstitutionCipher,
    get_bigrams,
    plausibility,
    prolom_substitute,
    substitute_decrypt,
    substitute_encrypt,
    transition_matrix,
)
from substitution_cipher.bigrams import load_matrix

KR_MATRIX_PATH = PROJECT_ROOT / 'data' / 'processed' / 'TM_ref_krakatit.npy'
COMBINED_MATRIX_PATH = PROJECT_ROOT / 'data' / 'processed' / 'TM_ref_combined.npy'
TM_ref = load_matrix(KR_MATRIX_PATH)

print('Projekt:', PROJECT_ROOT)
print('Matice:', KR_MATRIX_PATH.relative_to(PROJECT_ROOT))
print('Abeceda:', ALPHABET)


Projekt: D:\unicorn\python
Matice: data\processed\TM_ref_krakatit.npy
Abeceda: ABCDEFGHIJKLMNOPQRSTUVWXYZ_


## Povinné funkční API

PDF zadání vyžaduje samostatné funkce `substitute_encrypt`, `substitute_decrypt`, `get_bigrams`, `transition_matrix`, `plausibility` a `prolom_substitute`. Objektová třída `SubstitutionCipher` je pouze pohodlná fasáda nad stejnou implementací.


In [2]:
key = 'DEFGHIJKLMNOPQRSTUVWXYZ_ABC'
plaintext = 'BYL_POZDNI_VECER'

ciphertext = substitute_encrypt(plaintext, key)
decrypted = substitute_decrypt(ciphertext, key)

bigrams = get_bigrams(plaintext)
absolute_matrix = transition_matrix(bigrams)
relative_matrix = absolute_matrix / absolute_matrix.sum()
score = plausibility(plaintext, relative_matrix)

found_key, found_plaintext, found_score = prolom_substitute(
    ciphertext,
    relative_matrix,
    5,
    ALPHABET,
)

print('Plaintext:', plaintext)
print('Ciphertext:', ciphertext)
print('Decrypted:', decrypted)
print('Round-trip OK:', decrypted == plaintext)
print('Počet bigramů:', len(bigrams))
print('Matice:', relative_matrix.shape, 'suma', relative_matrix.sum())
print('Ukázka prolom_substitute:', found_plaintext, found_score)


Plaintext: BYL_POZDNI_VECER
Ciphertext: EAOCSRBGQLCYHFHU
Decrypted: BYL_POZDNI_VECER
Round-trip OK: True
Počet bigramů: 15
Matice: (27, 27) suma 1.0
Ukázka prolom_substitute: EAOCSRBGQLCYHFHU -98.87510598012987


## Objektové API

Doporučené použití je prostřednictvím třídy `SubstitutionCipher`. Následující buňka spouští pouze krátkou demonstrační kryptoanalýzu na jednom učitelském ciphertextu. Finální běh se provádí skriptem `scripts/decrypt_samples.py` s konfigurací `2 × 10 000`.


In [3]:
sample_ciphertext_path = PROJECT_ROOT / 'data' / 'ciphertexts' / 'text_1000_sample_1_ciphertext.txt'

if sample_ciphertext_path.exists():
    sample_ciphertext = sample_ciphertext_path.read_text(encoding='utf-8').strip()
    cipher = SubstitutionCipher.from_matrix_file(KR_MATRIX_PATH)
    demo_result = cipher.crack(
        sample_ciphertext,
        iterations=100,
        restarts=1,
        seed=1,
        polish=False,
        progress_every=0,
    )
    print('Délka ciphertextu:', len(sample_ciphertext))
    print('Začátek demonstračního plaintextu:', demo_result.plaintext[:120])
    print('Demonstrační skóre:', demo_result.score)
else:
    print('Ukázkový ciphertext není k dispozici.')


Restart 1/1 best plausibility: -6789.140269238431
Overall best restart: 1, plausibility: -6789.140269238431
Délka ciphertextu: 1000
Začátek demonstračního plaintextu: OUNV_ROANOD_JNOTOVTKORD_OM_VPOANOI_KCLOOVTIJU_STOK_OCTHOHYALOUEXTUNOD_MNOUS_UNOC_ALOZ_OHND_IOECTSOK_OCPK_OENHLUSTOJSTUNY
Demonstrační skóre: -6789.140269238431


## Referenční matice

Projekt uchovává samostatnou matici z Krakatitu, samostatnou matici z Války s mloky i kombinovanou matici z obou textů. Finální konfigurace zatím používá Krakatit, protože benchmark na kontrolním vzorku byl s touto maticí stabilní.

Nulové hodnoty mohou vzniknout v absolutní bigramové matici u dvojic znaků, které se v referenčním textu nevyskytly. Před normalizací se proto nahrazují hodnotou `1`, aby se při výpočtu věrohodnosti nepočítal logaritmus z nuly.


In [4]:
for name, path in [('krakatit', KR_MATRIX_PATH), ('combined', COMBINED_MATRIX_PATH)]:
    if path.exists():
        matrix = load_matrix(path)
        print(name)
        print('  shape:', matrix.shape)
        print('  suma:', float(matrix.sum()))
        print('  obsahuje nuly:', bool((matrix == 0).any()))
    else:
        print(name, 'matice chybí:', path)


krakatit
  shape: (27, 27)
  suma: 0.9999999999999999
  obsahuje nuly: False
combined
  shape: (27, 27)
  suma: 1.0
  obsahuje nuly: False


In [ ]:
plt.figure(figsize=(7, 6))
plt.imshow(np.log(TM_ref), cmap='viridis')
plt.colorbar(label='log pravděpodobnosti')
plt.xticks(range(len(ALPHABET)), list(ALPHABET), rotation=90)
plt.yticks(range(len(ALPHABET)), list(ALPHABET))
plt.title('Referenční bigramová matice: Krakatit')
plt.tight_layout()
plt.show()


## Výsledek benchmarku

Nejdůležitější naměřené výsledky pro známý učitelský kontrolní vzorek:

| Strategie | Matice | Průměr | Minimum | Přesné běhy |
|---|---|---:|---:|---:|
| 1 × 20 000 | combined | 94,80 % | 92,20 % | 1/3 |
| 1 × 20 000 | krakatit | 93,60 % | 90,00 % | 1/3 |
| 2 × 10 000 | krakatit | 100,00 % | 100,00 % | 3/3 |
| 5 × 4 000 | krakatit | 100,00 % | 100,00 % | 3/3 |

Vybrána byla strategie `2 × 10 000`, protože měla stejnou přesnost jako `5 × 4 000`, byla mírně rychlejší a její konfigurace je jednodušší. Učitelský plaintext a klíč byly použity pouze pro následné vyhodnocení benchmarku, nikoli během hledání.

Tento výsledek není univerzální zárukou 100% přesnosti na všech neznámých textech.


In [6]:
benchmark_csv = PROJECT_ROOT / 'reports' / 'search_strategy_benchmark.csv'
benchmark_md = PROJECT_ROOT / 'reports' / 'search_strategy_benchmark.md'

print('Benchmark Markdown existuje:', benchmark_md.exists())
print('Benchmark CSV existuje:', benchmark_csv.exists())

if benchmark_csv.exists():
    with benchmark_csv.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    print('Počet řádků benchmarku:', len(rows))
    if rows:
        best = max(rows, key=lambda row: float(row['matching_percent']))
        print('Nejlepší řádek:', best['strategy'], best['matrix'], best['matching_percent'])


Benchmark Markdown existuje: True
Benchmark CSV existuje: True
Počet řádků benchmarku: 30
Nejlepší řádek: 1x20000 krakatit 100.0


## Finální výsledky z `outputs/`

Notebook pouze načítá existující výstupy. Pokud jste ještě nespustili finální dešifrování, spusťte `run.bat` nebo příkaz:

```powershell
python scripts/decrypt_samples.py --matrix data\processed\TM_ref_krakatit.npy --iterations 10000 --restarts 2
```


In [7]:
outputs_dir = PROJECT_ROOT / 'outputs'
plaintext_files = sorted(outputs_dir.glob('*_plaintext.txt')) if outputs_dir.exists() else []
key_files = sorted(outputs_dir.glob('*_key.txt')) if outputs_dir.exists() else []

print('Plaintext soubory:', len(plaintext_files))
print('Key soubory:', len(key_files))

sample_plaintext_path = outputs_dir / 'text_1000_sample_1_plaintext.txt'
sample_key_path = outputs_dir / 'text_1000_sample_1_key.txt'

if sample_plaintext_path.exists() and sample_key_path.exists():
    sample_plaintext = sample_plaintext_path.read_text(encoding='utf-8').strip()
    sample_key = sample_key_path.read_text(encoding='utf-8').strip()
    print('Ukázka plaintextu:', sample_plaintext[:200])
    print('Délka klíče:', len(sample_key))
else:
    print('Výstup pro text_1000_sample_1 zatím není k dispozici.')


Plaintext soubory: 60
Key soubory: 60
Ukázka plaintextu: _VOZEM_DO_NEHO_A_ZAS_MNE_BEZI_DO_CESTY__ZACHVELA_SE_TAK_KUDY_VPRAVO_NEBO_VLEVO_TEDY_JE_KONEC_PTAL_SE_TISE_POKYVLA_HLAVOU_TEDY_JE_KONEC_OTEVREL_DVIRKA_VYSKOCIL_Z_VOZU_A_POSTAVIL_SE_PRED_KOLA_JED_REKL_C
Délka klíče: 27


## Vyhodnocení známého učitelského vzorku

Soubor `data/teacher_example/text_1000_sample_1_plaintext.txt` slouží pouze k vyhodnocení kvality výsledku. Během dešifrování není algoritmu dostupný.


In [8]:
teacher_plaintext_path = PROJECT_ROOT / 'data' / 'teacher_example' / 'text_1000_sample_1_plaintext.txt'
teacher_key_path = PROJECT_ROOT / 'data' / 'teacher_example' / 'text_1000_sample_1_key.txt'

if teacher_plaintext_path.exists() and sample_plaintext_path.exists():
    teacher_plaintext = teacher_plaintext_path.read_text(encoding='utf-8').strip()
    output_plaintext = sample_plaintext_path.read_text(encoding='utf-8').strip()
    matching_chars = sum(a == b for a, b in zip(output_plaintext, teacher_plaintext))
    print('Shodné znaky:', matching_chars, '/', len(teacher_plaintext))
    print('Shoda [%]:', 100 * matching_chars / len(teacher_plaintext))
    print('Plaintext přesný:', output_plaintext == teacher_plaintext)
else:
    print('Učitelský vzorek nebo výstup zatím není k dispozici.')

if teacher_key_path.exists() and sample_key_path.exists():
    teacher_key = teacher_key_path.read_text(encoding='utf-8').strip()
    output_key = sample_key_path.read_text(encoding='utf-8').strip()
    print('Klíč přesný:', output_key == teacher_key)


Shodné znaky: 1000 / 1000
Shoda [%]: 100.0
Plaintext přesný: True
Klíč přesný: False


In [9]:
evaluation_csv = PROJECT_ROOT / 'reports' / 'evaluation_summary.csv'
if evaluation_csv.exists():
    with evaluation_csv.open(encoding='utf-8', newline='') as handle:
        rows = list(csv.DictReader(handle))
    print('Počet řádků ve vyhodnocení:', len(rows))
    teacher_rows = [row for row in rows if row.get('length') == '1000' and row.get('sample_id') == '1']
    if teacher_rows:
        print('Řádek učitelského vzorku:')
        for key, value in teacher_rows[0].items():
            print(f'  {key}: {value}')
else:
    print('evaluation_summary.csv zatím neexistuje.')


Počet řádků ve vyhodnocení: 60
Řádek učitelského vzorku:
  length: 1000
  sample_id: 1
  plaintext_file: text_1000_sample_1_plaintext.txt
  key_file: text_1000_sample_1_key.txt
  plaintext_length: 1000
  key_valid: True
  plausibility: -5208.504696
  matches_teacher_example: True
  matching_chars: 1000
  matching_percent: 100.000000
  key_matches_teacher_example: False


## Omezení

Metropolis-Hastingsův algoritmus je náhodný a různé seedy mohou vést k různým výsledkům. Kratší texty jsou obtížnější, protože obsahují méně bigramů. Některé znaky se v ciphertextu nemusí objevit, takže celý klíč nemusí být jednoznačně určitelný. Bigramový model hodnotí jazykovou věrohodnost, nikoli skutečnou znalost správného plaintextu.
